In [30]:
import arcpy
import os
import openpyxl

In [31]:
# ============================================================
# CONFIGURATION
# ============================================================

excel_file = r"C:\National Water Utility\Domains.xlsx"
gdb = r"C:\National Water Utility\NWSC_Materials.gdb"
worksheet_name = "Domains"

In [32]:
# ============================================================
# VALIDATE INPUTS
# ============================================================

if not os.path.exists(excel_file):
    raise FileNotFoundError(
        f"Excel file not found:\n{excel_file}"
    )

if not arcpy.Exists(gdb):
    raise FileNotFoundError(
        f"Geodatabase not found:\n{gdb}"
    )


# ============================================================
# FUNCTION: CONVERT EXCEL CODE TO ARCGIS TYPE
# ============================================================

def convert_code(code, field_type):

    if code is None:
        return None

    field_type = field_type.upper().strip()

    # --------------------------------------------------------
    # SHORT
    # --------------------------------------------------------

    if field_type == "SHORT":

        try:
            return int(float(code))
        except:
            raise ValueError(
                f"Invalid SHORT domain code: '{code}'"
            )

    # --------------------------------------------------------
    # LONG
    # --------------------------------------------------------

    elif field_type == "LONG":

        try:
            return int(float(code))
        except:
            raise ValueError(
                f"Invalid LONG domain code: '{code}'"
            )

    # --------------------------------------------------------
    # FLOAT
    # --------------------------------------------------------

    elif field_type == "FLOAT":

        try:
            return float(code)
        except:
            raise ValueError(
                f"Invalid FLOAT domain code: '{code}'"
            )

    # --------------------------------------------------------
    # DOUBLE
    # --------------------------------------------------------

    elif field_type == "DOUBLE":

        try:
            return float(code)
        except:
            raise ValueError(
                f"Invalid DOUBLE domain code: '{code}'"
            )

    # --------------------------------------------------------
    # TEXT
    # --------------------------------------------------------

    elif field_type == "TEXT":

        return str(code).strip()

    else:

        raise ValueError(
            f"Unsupported ArcGIS field type: {field_type}"
        )


# ============================================================
# READ EXCEL
# ============================================================

print("Reading Excel workbook...")

workbook = openpyxl.load_workbook(
    excel_file,
    data_only=True
)

if worksheet_name not in workbook.sheetnames:
    raise ValueError(
        f"Worksheet '{worksheet_name}' does not exist."
    )

worksheet = workbook[worksheet_name]


# ============================================================
# READ HEADERS
# ============================================================

headers = [
    cell.value
    for cell in worksheet[1]
]

headers = [
    str(h).strip()
    if h is not None
    else ""
    for h in headers
]

required_headers = [
    "DomainName",
    "DomainDescription",
    "FieldType",
    "Code",
    "Description"
]

for header in required_headers:

    if header not in headers:

        raise ValueError(
            f"Missing required Excel column: {header}"
        )


header_index = {
    header: headers.index(header)
    for header in required_headers
}


# ============================================================
# READ DOMAIN DEFINITIONS
# ============================================================

domains = {}

for row_number, row in enumerate(
    worksheet.iter_rows(
        min_row=2,
        values_only=True
    ),
    start=2
):

    domain_name = row[
        header_index["DomainName"]
    ]

    domain_description = row[
        header_index["DomainDescription"]
    ]

    field_type = row[
        header_index["FieldType"]
    ]

    code = row[
        header_index["Code"]
    ]

    description = row[
        header_index["Description"]
    ]

    # Skip empty rows

    if domain_name is None:
        continue

    domain_name = str(
        domain_name
    ).strip()

    field_type = str(
        field_type
    ).strip().upper()

    domain_description = (
        str(domain_description).strip()
        if domain_description
        else domain_name
    )

    # --------------------------------------------------------
    # Validate field type
    # --------------------------------------------------------

    valid_types = [
        "SHORT",
        "LONG",
        "FLOAT",
        "DOUBLE",
        "TEXT"
    ]

    if field_type not in valid_types:

        raise ValueError(
            f"Row {row_number}: "
            f"Invalid field type '{field_type}'. "
            f"Allowed values: {valid_types}"
        )

    # --------------------------------------------------------
    # Convert code
    # --------------------------------------------------------

    converted_code = convert_code(
        code,
        field_type
    )

    if converted_code is None:
        continue

    # --------------------------------------------------------
    # Store domain
    # --------------------------------------------------------

    if domain_name not in domains:

        domains[domain_name] = {
            "description": domain_description,
            "field_type": field_type,
            "values": []
        }

    domains[domain_name]["values"].append(
        (
            converted_code,
            str(description)
            if description is not None
            else str(converted_code)
        )
    )


# ============================================================
# GET EXISTING DOMAINS
# ============================================================

existing_domains = {
    d.name: d
    for d in arcpy.da.ListDomains(gdb)
}


# ============================================================
# CREATE DOMAINS
# ============================================================

print("\n========================================")
print("CREATING DOMAINS")
print("========================================\n")

for domain_name, domain_info in domains.items():

    field_type = domain_info["field_type"]
    description = domain_info["description"]

    print(
        f"Domain: {domain_name}"
    )

    print(
        f"Type: {field_type}"
    )

    # --------------------------------------------------------
    # Check if domain exists
    # --------------------------------------------------------

    if domain_name in existing_domains:

        print(
            "  Domain already exists."
        )

        continue

    # --------------------------------------------------------
    # Create domain
    # --------------------------------------------------------

    arcpy.management.CreateDomain(
        in_workspace=gdb,
        domain_name=domain_name,
        domain_description=description,
        field_type=field_type,
        domain_type="CODED",
        split_policy="DEFAULT",
        merge_policy="DEFAULT"
    )

    print(
        "  Domain created."
    )

    # --------------------------------------------------------
    # Add coded values
    # --------------------------------------------------------

    for code, code_description in domain_info["values"]:

        try:

            arcpy.management.AddCodedValueToDomain(
                in_workspace=gdb,
                domain_name=domain_name,
                code=code,
                code_description=code_description
            )

            print(
                f"  Added: {code} = {code_description}"
            )

        except Exception as e:

            print(
                f"  ERROR adding code "
                f"'{code}': {e}"
            )


print("\n========================================")
print("PROCESS COMPLETED")
print("========================================")

print(
    f"Domains processed: {len(domains)}"
)

print(
    f"Geodatabase: {gdb}"
)

Reading Excel workbook...

CREATING DOMAINS

Domain: Type_of_Material
Type: SHORT
  Domain created.
  Added: 1 = Adaptor
  Added: 2 = Airvalve
  Added: 3 = Bend
  Added: 4 = Bolts and Nuts
  Added: 5 = Concrete Ring
  Added: 6 = Concrete Slub
  Added: 7 = Coupling
  Added: 8 = Cover
  Added: 9 = Cutting Disc
  Added: 10 = Elbow
  Added: 11 = End Cap
  Added: 12 = Flange
  Added: 13 = Gasket
  Added: 14 = Grinding Disc
  Added: 15 = Hydrant
  Added: 16 = Nipple
  Added: 17 = Pipe
  Added: 18 = Plug
  Added: 19 = Reducer
  Added: 20 = Reinforcement Bar
  Added: 21 = Repair Clamp
  Added: 22 = Saddle
  Added: 23 = Socket
  Added: 24 = Stepiron
  Added: 25 = Stop Cock
  Added: 26 = Straight Connector
  Added: 27 = Strainer
  Added: 28 = Tee Piece
  Added: 29 = Valve
  Added: 30 = Water Meter
  Added: 31 = Welding Rod
Domain: Pipe_sizes
Type: SHORT
  Domain created.
  Added: 32 = 1/8” (DN6)
  Added: 33 = ¼” (DN8)
  Added: 34 = 3/8” (DN10)
  Added: 35 = ½” (DN15)
  Added: 36 = ¾” (DN20)
  Ad